# Асинхронный веб-краулер

### Установка зависимостей

In [1]:
%pip install aiohttp aiofiles beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


### Импорты
Все библиотеки, которые используются в ноутбуке, собраны здесь.

In [2]:
import asyncio
import aiohttp
import logging
import time
import json
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

### Настройка логирования

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("crawler")

## Класс HTMLParser

In [4]:
class HTMLParser:
    def empty_result(self, url: str) -> dict:
        return {
            "url": url,
            "title": "",
            "text": "",
            "links": [],
            "metadata": {},
            "images": [],
            "headings": {},
            "tables": [],
            "lists": [],
        }

    async def parse_html(self, html: str, url: str) -> dict:
        result = self.empty_result(url)
        try:
            soup = await asyncio.to_thread(BeautifulSoup, html, "lxml")
        except Exception as e:
            logger.warning("Не удалось разобрать HTML (%s): %s", type(e).__name__, url)
            return result

        extractors = {
            "metadata": lambda: self.extract_metadata(soup),
            "text": lambda: self.extract_text(soup),
            "links": lambda: self.extract_links(soup, url),
            "images": lambda: self.extract_images(soup, url),
            "headings": lambda: self.extract_headings(soup),
            "tables": lambda: self.extract_tables(soup),
            "lists": lambda: self.extract_lists(soup),
        }
        for field, extract in extractors.items():
            try:
                result[field] = extract()
            except Exception as e:
                logger.warning(
                    "Ошибка при извлечении %s (%s): %s", field, type(e).__name__, url
                )

        result["title"] = result["metadata"].get("title", "")
        return result

    def extract_links(self, soup: BeautifulSoup, base_url: str) -> list[str]:
        links = []
        for tag in soup.find_all("a", href=True):
            absolute_url = urljoin(base_url, tag["href"])
            if self.is_valid_url(absolute_url):
                links.append(absolute_url)
        return links

    def is_valid_url(self, url: str) -> bool:
        parsed = urlparse(url)
        return parsed.scheme in ("http", "https") and bool(parsed.netloc)

    def extract_text(self, soup: BeautifulSoup, selector: str = None) -> str:
        root = soup.select_one(selector) if selector else soup
        if root is None:
            return ""
        return root.get_text(separator=" ", strip=True)

    def extract_metadata(self, soup: BeautifulSoup) -> dict:
        def meta_content(name: str) -> str:
            tag = soup.find("meta", attrs={"name": name})
            return tag.get("content", "") if tag else ""

        title = soup.title.get_text(strip=True) if soup.title else ""
        return {
            "title": title,
            "description": meta_content("description"),
            "keywords": meta_content("keywords"),
        }
    
    def extract_images(self, soup: BeautifulSoup, base_url: str) -> list[dict]:
        images = []
        for tag in soup.find_all("img", src=True):
            images.append({
                "src": urljoin(base_url, tag["src"]),
                "alt": tag.get("alt", ""),
            })
        return images

    def extract_headings(self, soup: BeautifulSoup) -> dict[str, list[str]]:
        headings = {}
        for level in ("h1", "h2", "h3"):
            headings[level] = [
                tag.get_text(strip=True) for tag in soup.find_all(level)
            ]
        return headings

    def extract_tables(self, soup: BeautifulSoup) -> list[list[list[str]]]:
        tables = []
        for table in soup.find_all("table"):
            rows = []
            for tr in table.find_all("tr"):
                cells = [
                    cell.get_text(strip=True) for cell in tr.find_all(["th", "td"])
                ]
                if cells:
                    rows.append(cells)
            tables.append(rows)
        return tables

    def extract_lists(self, soup: BeautifulSoup) -> list[dict]:
        lists = []
        for tag in soup.find_all(["ul", "ol"]):
            items = [
                li.get_text(strip=True) for li in tag.find_all("li", recursive=False)
            ]
            lists.append({"type": tag.name, "items": items})
        return lists

## Класс AsyncCrawler
Сессия создаётся один раз и переиспользуется для всех запросов.
Семафор ограничивает число одновременных запросов значением max_concurrent.

In [5]:
class AsyncCrawler:
    def __init__(
        self,
        max_concurrent: int = 10,
        connect_timeout: float = 5,
        read_timeout: float = 10, 
    ):
        self.max_concurrent = max_concurrent
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.parser = HTMLParser()

        timeout = aiohttp.ClientTimeout(
            connect=connect_timeout,
            sock_read=read_timeout,
        )
        # используется системный поиск адресов, чтобы краулер работал через VPN
        connector = aiohttp.TCPConnector(
            limit=max_concurrent,
            resolver=aiohttp.ThreadedResolver(),
        )

        self.session = aiohttp.ClientSession(
            timeout=timeout,
            connector=connector,
        )

    async def fetch_url(self, url: str) -> str | None:
        async with self.semaphore:
            logger.info("Начало загрузки: %s", url)
            try:
                async with self.session.get(url) as response:
                    response.raise_for_status()
                    text = await response.text()
                    logger.info(
                        "Успешно: %s (статус %s, %d символов)",
                        url, response.status, len(text),
                    )
                    return text
            except aiohttp.ClientResponseError as e:
                logger.error("%s, статус %s: %s", type(e).__name__, e.status, url)
            except asyncio.TimeoutError as e:
                logger.error("%s, таймаут: %s", type(e).__name__, url)
            except aiohttp.ClientError as e:
                logger.error("%s, сетевая ошибка: %s", type(e).__name__, url)
            return None

    async def fetch_urls(self, urls: list[str]) -> dict[str, str]:
        tasks = [self.fetch_url(url) for url in urls]
        pages = await asyncio.gather(*tasks)
        return {url: page for url, page in zip(urls, pages) if page is not None}

    async def fetch_and_parse(self, url: str) -> dict:
        html = await self.fetch_url(url)
        if html is None:
            return self.parser.empty_result(url)
        return await self.parser.parse_html(html, url)

    async def close(self):
        await self.session.close()

## Демонстрация 1 дня

In [6]:
DEMO_URLS = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/get",
    "https://httpbin.org/html",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/2",
    "https://httpbin.org/delay/3",
    "https://httpbin.org/status/404",
]


async def run_sequential(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    results = {}
    start = time.perf_counter()
    try:
        for url in urls:
            page = await crawler.fetch_url(url)
            if page is not None:
                results[url] = page
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


async def run_parallel(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        results = await crawler.fetch_urls(urls)
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


def print_report(title: str, urls: list[str], results: dict[str, str], elapsed: float):
    print(f"\n{title}")
    for url in urls:
        mark = "✅" if url in results else "❌"
        print(f"  {mark} {url}")
    print(f"  Загружено {len(results)} из {len(urls)} за {elapsed:.2f} сек")


async def main():
    seq_results, seq_time = await run_sequential(DEMO_URLS)
    par_results, par_time = await run_parallel(DEMO_URLS)

    print_report("Последовательная загрузка", DEMO_URLS, seq_results, seq_time)
    print_report("Параллельная загрузка", DEMO_URLS, par_results, par_time)
    print(f"\nПараллельно быстрее в {seq_time / par_time:.1f} раза")


await main()

16:32:41 [INFO] Начало загрузки: https://example.com
16:32:41 [INFO] Успешно: https://example.com (статус 200, 577 символов)
16:32:41 [INFO] Начало загрузки: https://www.python.org
16:32:42 [INFO] Успешно: https://www.python.org (статус 200, 52249 символов)
16:32:42 [INFO] Начало загрузки: https://httpbin.org/get
16:32:43 [INFO] Успешно: https://httpbin.org/get (статус 200, 322 символов)
16:32:43 [INFO] Начало загрузки: https://httpbin.org/html
16:32:43 [INFO] Успешно: https://httpbin.org/html (статус 200, 3739 символов)
16:32:43 [INFO] Начало загрузки: https://httpbin.org/delay/1
16:32:44 [INFO] Успешно: https://httpbin.org/delay/1 (статус 200, 372 символов)
16:32:44 [INFO] Начало загрузки: https://httpbin.org/delay/2
16:32:46 [INFO] Успешно: https://httpbin.org/delay/2 (статус 200, 372 символов)
16:32:46 [INFO] Начало загрузки: https://httpbin.org/delay/3
16:32:49 [INFO] Успешно: https://httpbin.org/delay/3 (статус 200, 372 символов)
16:32:49 [INFO] Начало загрузки: https://httpbin.o


Последовательная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 8.42 сек

Параллельная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 3.96 сек

Параллельно быстрее в 2.1 раза


## Тестирование 1 дня

In [7]:
async def test_valid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://example.com",
            "https://httpbin.org/html",
        ])
    finally:
        await crawler.close()
    assert len(results) == 2, f"ожидали 2 страницы, получили {len(results)}"
    assert "Example Domain" in results["https://example.com"], "не тот текст страницы"


async def test_invalid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://this-site-does-not-exist-12345.com",
            "https://httpbin.org/status/404",
        ])
    finally:
        await crawler.close()
    assert results == {}, f"ожидали пустой результат, получили {list(results)}"


async def test_timeout():
    crawler = AsyncCrawler(read_timeout=1)
    start = time.perf_counter()
    try:
        page = await crawler.fetch_url("https://httpbin.org/delay/5")
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start
    assert page is None, "страница не должна была загрузиться"
    assert elapsed < 4, f"таймаут не сработал, ждали {elapsed:.1f} сек"


async def test_parallel_faster():
    urls = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/delay/3",
    ]
    _, seq_time = await run_sequential(urls)
    _, par_time = await run_parallel(urls)
    assert par_time < seq_time * 0.75, (
        f"параллельно {par_time:.1f} сек, последовательно {seq_time:.1f} сек"
    )


async def run_tests(tests: list):
    logger.setLevel(logging.WARNING)
    passed = 0
    try:
        for test in tests:
            try:
                await test()
                print(f"✅ {test.__name__}")
                passed += 1
            except AssertionError as e:
                print(f"❌ {test.__name__}: {e}")
    finally:
        logger.setLevel(logging.INFO)
    print(f"\nПройдено {passed} из {len(tests)}")


await run_tests([test_valid_urls, test_invalid_urls, test_timeout, test_parallel_faster])

16:32:54 [ERROR] ClientConnectorDNSError, сетевая ошибка: https://this-site-does-not-exist-12345.com


✅ test_valid_urls


16:32:55 [ERROR] ClientResponseError, статус 404: https://httpbin.org/status/404


✅ test_invalid_urls


16:32:56 [ERROR] SocketTimeoutError, таймаут: https://httpbin.org/delay/5


✅ test_timeout
✅ test_parallel_faster

Пройдено 4 из 4


## Демонстрация 2 дня

In [8]:
DEMO_PAGES = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/html",
    "https://docs.python.org/3/",
]


def summarize(page: dict) -> dict:
    return {
        "url": page["url"],
        "title": page["title"],
        "h1": page["headings"].get("h1", []),
        "text_length": len(page["text"]),
        "links_count": len(page["links"]),
        "links": page["links"][:5],
        "images_count": len(page["images"]),
    }


async def demo_parsing(urls: list[str]):
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        pages = await asyncio.gather(*(crawler.fetch_and_parse(url) for url in urls))
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start

    for page in pages:
        print(json.dumps(summarize(page), ensure_ascii=False, indent=2))

    total_links = sum(len(page["links"]) for page in pages)
    total_text = sum(len(page["text"]) for page in pages)
    print("\nСтатистика")
    print(f"  Страниц: {len(pages)}")
    print(f"  Всего ссылок: {total_links}")
    print(f"  Общая длина текста: {total_text} символов")
    print(f"  Время: {elapsed:.2f} сек")


await demo_parsing(DEMO_PAGES)

16:33:07 [INFO] Начало загрузки: https://example.com
16:33:07 [INFO] Начало загрузки: https://www.python.org
16:33:08 [INFO] Начало загрузки: https://httpbin.org/html
16:33:08 [INFO] Начало загрузки: https://docs.python.org/3/
16:33:08 [INFO] Успешно: https://docs.python.org/3/ (статус 200, 19620 символов)
16:33:08 [INFO] Успешно: https://example.com (статус 200, 577 символов)
16:33:08 [INFO] Успешно: https://www.python.org (статус 200, 52249 символов)
16:33:09 [INFO] Успешно: https://httpbin.org/html (статус 200, 3739 символов)


{
  "url": "https://example.com",
  "title": "Example Domain",
  "h1": [],
  "text_length": 171,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://www.python.org",
  "title": "Welcome to Python.org",
  "h1": [
    "Functions Defined",
    "All the Flow You’d Expect",
    "Compound Data Types",
    "Intuitive Interpretation",
    "Quick & Easy to Learn"
  ],
  "text_length": 6952,
  "links_count": 212,
  "links": [
    "https://www.python.org#content",
    "https://www.python.org#python-network",
    "https://www.python.org/",
    "https://www.python.org/psf/",
    "https://docs.python.org"
  ],
  "images_count": 1
}
{
  "url": "https://httpbin.org/html",
  "title": "",
  "h1": [
    "Herman Melville - Moby-Dick"
  ],
  "text_length": 3594,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://docs.python.org/3/",
  "title": "3.15.0 Documentation",
  "h1": [
    "Python 3.15.0 documentation"
  ],
  "text_length": 3217,
  "links_count": 9

## Тестирование 2 дня

In [9]:
VALID_HTML = """
<html>
  <head>
    <title>Тестовая страница</title>
    <meta name="description" content="Описание страницы">
  </head>
  <body>
    <h1>Главный заголовок</h1>
    <p>Основной текст страницы.</p>
    <img src="/logo.png" alt="Логотип">
    <table><tr><th>Город</th></tr><tr><td>Москва</td></tr></table>
    <ul><li>Чай</li><li>Кофе</li></ul>
  </body>
</html>
"""


async def test_parse_valid_html():
    result = await HTMLParser().parse_html(VALID_HTML, "https://example.com")
    assert result["title"] == "Тестовая страница", f"title: {result['title']}"
    assert result["metadata"]["description"] == "Описание страницы", "нет description"
    assert result["headings"]["h1"] == ["Главный заголовок"], "неверный h1"
    assert "Основной текст страницы." in result["text"], "нет текста"
    assert result["images"] == [
        {"src": "https://example.com/logo.png", "alt": "Логотип"}
    ], "неверные картинки"
    assert result["tables"] == [[["Город"], ["Москва"]]], "неверная таблица"
    assert result["lists"] == [{"type": "ul", "items": ["Чай", "Кофе"]}], "неверный список"


async def test_parse_broken_html():
    parser = HTMLParser()
    broken = "<title>Битая</title><h1>Заголовок<p>Текст<a href='/page'>ссылка<table><tr><td>A<td>B"
    result = await parser.parse_html(broken, "https://example.com")
    assert result["title"] == "Битая", f"title: {result['title']}"
    assert result["links"] == ["https://example.com/page"], f"links: {result['links']}"
    assert result["tables"] == [[["A", "B"]]], f"tables: {result['tables']}"

    result = await parser.parse_html(None, "https://example.com")
    assert result == parser.empty_result("https://example.com"), "ожидали пустой результат"


async def test_extract_links():
    html = """
    <a href="https://python.org">1</a>
    <a href="mailto:info@example.com">2</a>
    <a href="javascript:void(0)">3</a>
    <a href="tel:+70000000000">4</a>
    <a>без href</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com")
    assert links == ["https://python.org"], f"links: {links}"


async def test_relative_urls():
    html = """
    <a href="/about">1</a>
    <a href="page2.html">2</a>
    <a href="../archive">3</a>
    <a href="//cdn.example.org/lib">4</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com/news/today")
    assert links == [
        "https://example.com/about",
        "https://example.com/news/page2.html",
        "https://example.com/archive",
        "https://cdn.example.org/lib",
    ], f"links: {links}"


await run_tests([test_parse_valid_html, test_parse_broken_html, test_extract_links, test_relative_urls])

16:33:09 [WARNING] Не удалось разобрать HTML (TypeError): https://example.com


✅ test_parse_valid_html
✅ test_parse_broken_html
✅ test_extract_links
✅ test_relative_urls

Пройдено 4 из 4
